In [1]:
# ============================================================
# CHECKPOINT 1 — Environment setup
# Static vs. Dynamic Spectral Attention Topology project
# ============================================================

In [2]:
# --- Cell 1: GPU check ---
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("Memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)
else:
    print("WARNING: No GPU detected. In Kaggle: Settings > Accelerator > GPU T4 x2 (or P100).")

CUDA available: True
Device: Tesla T4
Memory (GB): 15.636037632


In [3]:
# --- Cell 2: internet check (needed for HF model/dataset downloads) ---
# In Kaggle: Settings > Internet > On (must be toggled per-session)
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=5)
    print("Internet: OK")
except Exception as e:
    print("WARNING: No internet access.", e)
    print("Fix: Notebook Settings (right panel) > Internet > toggle ON, then restart session.")

Internet: OK


In [4]:
# --- Cell 3: package install ---
# transformers/datasets versions matter for attention-output behavior; pin what your pilot validated.
!pip install -q -U transformers datasets accelerate scipy scikit-learn statsmodels

import transformers, datasets
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 26.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.0 which is incompatible.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.
transformers: 5.15.0
datasets: 5.0.1


In [5]:
# --- Cell 4: working directory + checkpoint structure ---
import os
BASE_DIR = "/kaggle/working/spectral_v2"
DIRS = {
    "generations": f"{BASE_DIR}/generations",     # raw model outputs per dataset
    "extractions": f"{BASE_DIR}/extractions",     # attention/hidden-state tensors
    "features":    f"{BASE_DIR}/features",        # computed spectral + dynamic features
    "results":     f"{BASE_DIR}/results",         # stats output, figures
    "logs":        f"{BASE_DIR}/logs",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
print("Directory structure created:")
for k, v in DIRS.items():
    print(f"  {k}: {v}")

Directory structure created:
  generations: /kaggle/working/spectral_v2/generations
  extractions: /kaggle/working/spectral_v2/extractions
  features: /kaggle/working/spectral_v2/features
  results: /kaggle/working/spectral_v2/results
  logs: /kaggle/working/spectral_v2/logs


In [6]:
# --- Cell 5: GPU quota sanity note ---
print("""
REMINDER: Kaggle GPU quota is 30 hrs/week (resets weekly, check My Account > quota).
Check remaining quota now before running anything expensive.
This pipeline should checkpoint after every stage (generations, extractions, features)
so you never have to redo expensive GPU work if the session disconnects.
""")


REMINDER: Kaggle GPU quota is 30 hrs/week (resets weekly, check My Account > quota).
Check remaining quota now before running anything expensive.
This pipeline should checkpoint after every stage (generations, extractions, features)
so you never have to redo expensive GPU work if the session disconnects.



In [7]:
# ============================================================
# CHECKPOINT 2 — Model load + attention support verification
# ============================================================

In [8]:
# --- Cell 1: load model ---
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# attn_implementation="eager" is REQUIRED — SDPA/FlashAttention silently
# return None for attentions instead of erroring, which will corrupt
# everything downstream without any warning. Do not change this.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    attn_implementation="eager",
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print("Model loaded:", MODEL_ID)
print("Num layers:", model.config.num_hidden_layers)
print("Num attention heads:", model.config.num_attention_heads)
print("Attn implementation:", model.config._attn_implementation)
assert model.config._attn_implementation == "eager", "STOP: attn_implementation did not take effect."

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-0.5B-Instruct
Num layers: 24
Num attention heads: 14
Attn implementation: eager


In [9]:
# --- Cell 2: verify chat template ---
test_messages = [{"role": "user", "content": "What is 12 + 7?"}]
prompt = tokenizer.apply_chat_template(
    test_messages, tokenize=False, add_generation_prompt=True
)
print("Chat template output:")
print(repr(prompt))
assert len(prompt) > 0, "STOP: chat template returned empty string."

Chat template output:
'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is 12 + 7?<|im_end|>\n<|im_start|>assistant\n'


In [10]:
# --- Cell 3: verify attention output shape on a real forward pass ---
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True, output_hidden_states=True)

# Sanity checks — if any of these fail, do not proceed to Checkpoint 3.
assert outputs.attentions is not None, "STOP: attentions is None — eager mode not actually active."
assert len(outputs.attentions) == model.config.num_hidden_layers, "STOP: layer count mismatch."

seq_len = inputs["input_ids"].shape[1]
first_layer_attn = outputs.attentions[0]
print("Attention tensor shape (layer 0):", first_layer_attn.shape)
print("Expected: [batch=1, heads={}, seq={}, seq={}]".format(
    model.config.num_attention_heads, seq_len, seq_len
))
assert first_layer_attn.shape == (1, model.config.num_attention_heads, seq_len, seq_len), \
    "STOP: attention shape doesn't match expected [1, heads, seq, seq]."

# Check attention rows sum to ~1 (row-stochastic, post-softmax)
row_sums = first_layer_attn[0, 0].sum(dim=-1)
print("Row sums (should be ~1.0):", row_sums[:5])
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-2), \
    "STOP: attention rows don't sum to 1 — something upstream is wrong."

print("\nAll Checkpoint 2 assertions passed.")
print(f"Layers: {model.config.num_hidden_layers}, Heads: {model.config.num_attention_heads}, Seq len (test): {seq_len}")

Attention tensor shape (layer 0): torch.Size([1, 14, 38, 38])
Expected: [batch=1, heads=14, seq=38, seq=38]
Row sums (should be ~1.0): tensor([1., 1., 1., 1., 1.], device='cuda:0', dtype=torch.float16)

All Checkpoint 2 assertions passed.
Layers: 24, Heads: 14, Seq len (test): 38


In [11]:
# --- Cell 4: quick generation smoke test ---
gen_out = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.3,
    pad_token_id=tokenizer.eos_token_id,
)
decoded = tokenizer.decode(gen_out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Test generation output:")
print(decoded)

# ============================================================
print("\n>>> If everything above passed: click Save Version now, then move to Checkpoint 3. <<<")

Test generation output:
12 + 7 equals 19.

>>> If everything above passed: click Save Version now, then move to Checkpoint 3. <<<


In [12]:
# ============================================================
# CHECKPOINT 3 — Teacher-forced full-sequence replay
# ============================================================
# Why this step exists: model.generate() does NOT give you clean
# attention tensors over the full prompt+response sequence in a
# usable form. So we generate first, then replay the exact same
# token IDs through a second forward pass with output_attentions=True.
# This is the step your pilot README calls out explicitly as
# checkpoint 3 in the original 9-checkpoint structure.

In [13]:
# --- Cell 1: full pipeline function, single example ---
import torch

def generate_and_replay(prompt_text, model, tokenizer, max_new_tokens=512, temperature=0.3):
    """
    Returns a dict with:
      - full_input_ids: prompt + generated tokens, concatenated
      - attentions: tuple of (num_layers) tensors, each [1, heads, seq, seq]
      - hidden_states: tuple of (num_layers+1) tensors, each [1, seq, d_model]
      - prompt_len, response_len, total_len
    """
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    # Step A: generate (no attention output here — generate() attention
    # outputs are awkward to work with across steps, so we don't use them)
    with torch.no_grad():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id,
        )

    full_input_ids = gen_ids  # [1, prompt_len + response_len]
    total_len = full_input_ids.shape[1]
    response_len = total_len - prompt_len

    # Step B: teacher-forced replay — full sequence, single forward pass,
    # with output_attentions=True and output_hidden_states=True.
    with torch.no_grad():
        replay_out = model(
            input_ids=full_input_ids,
            output_attentions=True,
            output_hidden_states=True,
        )

    return {
        "full_input_ids": full_input_ids,
        "attentions": replay_out.attentions,       # len = num_layers
        "hidden_states": replay_out.hidden_states,  # len = num_layers + 1
        "prompt_len": prompt_len,
        "response_len": response_len,
        "total_len": total_len,
    }

In [14]:
# --- Cell 2: run on one real example and validate shapes ---
test_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "If a train travels 60 miles in 1.5 hours, what is its average speed in miles per hour? Think step by step."}],
    tokenize=False, add_generation_prompt=True,
)

result = generate_and_replay(test_prompt, model, tokenizer)

print("Prompt length:", result["prompt_len"])
print("Response length:", result["response_len"])
print("Total length:", result["total_len"])
print("Num attention layers:", len(result["attentions"]))
print("Num hidden state layers:", len(result["hidden_states"]))
print("Attention shape (layer 0):", result["attentions"][0].shape)
print("Hidden state shape (layer 0):", result["hidden_states"][0].shape)

# Assertions — do not proceed if any of these fail
n_layers = model.config.num_hidden_layers
n_heads = model.config.num_attention_heads
total_len = result["total_len"]

assert len(result["attentions"]) == n_layers, "STOP: attention layer count mismatch."
assert len(result["hidden_states"]) == n_layers + 1, "STOP: hidden state layer count mismatch (should be n_layers+1, includes embedding output)."
assert result["attentions"][0].shape == (1, n_heads, total_len, total_len), "STOP: attention shape wrong — check it covers the FULL sequence, not just the response."
assert result["hidden_states"][0].shape == (1, total_len, model.config.hidden_size), "STOP: hidden state shape wrong."

# Check the replayed tokens match what was generated (teacher-forcing sanity check)
decoded_response = tokenizer.decode(
    result["full_input_ids"][0][result["prompt_len"]:], skip_special_tokens=True
)
print("\nDecoded response (from the generated tokens):")
print(decoded_response)
print("\nAll Checkpoint 3 assertions passed.")

Prompt length: 59
Response length: 148
Total length: 207
Num attention layers: 24
Num hidden state layers: 25
Attention shape (layer 0): torch.Size([1, 14, 207, 207])
Hidden state shape (layer 0): torch.Size([1, 207, 896])

Decoded response (from the generated tokens):
To calculate the average speed of the train, you can use the formula for average speed:

\[
\text{Average Speed} = \frac{\text{Total Distance}}{\text{Total Time}}
\]

Given:
- Total distance traveled: 60 miles
- Total time taken: 1.5 hours

Now, plug these values into the formula:

\[
\text{Average Speed} = \frac{60 \text{ miles}}{1.5 \text{ hours}}
\]

Perform the division:

\[
\text{Average Speed} = 40 \text{ miles per hour}
\]

So, the average speed of the train is **40 miles per hour**.

All Checkpoint 3 assertions passed.


In [15]:
# --- Cell 3: verify attention is causal (upper triangle should be ~0) ---
# This catches silent masking bugs — if this fails, attention weights are
# leaking future-token information, which would invalidate everything downstream.
attn_layer0_head0 = result["attentions"][0][0, 0]  # [seq, seq]
upper_triangle_mass = torch.triu(attn_layer0_head0, diagonal=1).sum().item()
print(f"\nUpper-triangle (future-token) attention mass: {upper_triangle_mass:.6f}")
assert upper_triangle_mass < 1e-4, "STOP: non-trivial attention to future tokens — causal masking is broken."
print("Causal masking verified correct.")


Upper-triangle (future-token) attention mass: 0.000000
Causal masking verified correct.


In [16]:
# ============================================================
# CHECKPOINT 4 — Graph construction
# Symmetrize, aggregate heads, build Laplacians (combinatorial + normalized)
# ============================================================

In [17]:
# --- Cell 1 ---
import numpy as np
import torch

def build_layer_graphs(attentions, eps=1e-8):
    """
    Takes the tuple of per-layer attention tensors from Checkpoint 3
    (each [1, heads, seq, seq]) and builds, for each layer:
      - symmetrized, head-aggregated weighted adjacency W
      - combinatorial Laplacian L = D - W
      - normalized (symmetric) Laplacian L_sym = I - D^-1/2 W D^-1/2

    Returns a list of dicts, one per layer.
    """
    n_layers = len(attentions)
    layer_graphs = []

    for l in range(n_layers):
        A = attentions[l][0]  # [heads, seq, seq], drop batch dim
        heads, seq, _ = A.shape

        # Step 1: per-head symmetrization
        # W^(l,h) = 0.5 * (A^(l,h) + A^(l,h)^T)
        A_np = A.detach().cpu().numpy().astype(np.float64)  # float64 for eigendecomposition stability
        W_per_head = 0.5 * (A_np + A_np.transpose(0, 2, 1))  # [heads, seq, seq]

        # Step 2: attention-mass-weighted head aggregation
        # weight each head by its total attention mass (sum of all entries),
        # matching the pilot's method (empirically ~uniform for standard dense softmax attn)
        head_mass = W_per_head.sum(axis=(1, 2))  # [heads]
        head_weights = head_mass / (head_mass.sum() + eps)
        W = np.tensordot(head_weights, W_per_head, axes=(0, 0))  # [seq, seq]
        
        # Step 3: combinatorial Laplacian
        D = np.diag(W.sum(axis=1))
        L = D - W

        # Step 4: normalized (symmetric) Laplacian — for Step 12 sensitivity check
        d = W.sum(axis=1)
        d_inv_sqrt = np.where(d > eps, 1.0 / np.sqrt(d + eps), 0.0)
        D_inv_sqrt = np.diag(d_inv_sqrt)
        L_sym = np.eye(seq) - D_inv_sqrt @ W @ D_inv_sqrt

        layer_graphs.append({
            "layer": l,
            "W": W,
            "L": L,
            "L_sym": L_sym,
            "head_weights": head_weights,
        })

    return layer_graphs

In [18]:
# --- Cell 2: run on the Checkpoint 3 test example and validate invariants ---
layer_graphs = build_layer_graphs(result["attentions"])

print(f"Built graphs for {len(layer_graphs)} layers.")
print(f"Graph size (num tokens): {layer_graphs[0]['W'].shape[0]}")

# Validate invariants on layer 0 and the last layer
for idx in [0, len(layer_graphs) - 1]:
    g = layer_graphs[idx]
    L = g["L"]
    L_sym = g["L_sym"]

    # L must be symmetric
    assert np.allclose(L, L.T, atol=1e-6), f"STOP: L not symmetric at layer {idx}."

    # L must be PSD -> smallest eigenvalue ~0, all eigenvalues >= -tiny tolerance
    eigvals_L = np.linalg.eigvalsh(L)
    print(f"Layer {idx}: smallest eigenvalue of L = {eigvals_L[0]:.8f} (should be ~0)")
    assert eigvals_L[0] > -1e-6, f"STOP: L has a meaningfully negative eigenvalue at layer {idx} — not PSD."
    assert eigvals_L.min() < 1e-4, f"STOP: smallest eigenvalue of L is not near zero at layer {idx}."

    # L_sym eigenvalues should lie in [0, 2]
    eigvals_Lsym = np.linalg.eigvalsh(L_sym)
    print(f"Layer {idx}: L_sym eigenvalue range = [{eigvals_Lsym.min():.4f}, {eigvals_Lsym.max():.4f}] (should be within [0, 2])")
    assert eigvals_Lsym.min() > -1e-6, f"STOP: L_sym has negative eigenvalue at layer {idx}."
    assert eigvals_Lsym.max() < 2 + 1e-6, f"STOP: L_sym eigenvalue exceeds 2 at layer {idx}."

    # head weights should sum to 1
    assert np.isclose(g["head_weights"].sum(), 1.0, atol=1e-6), f"STOP: head weights don't sum to 1 at layer {idx}."

print("\nAll Checkpoint 4 assertions passed — graphs are valid at layer 0 and the final layer.")

Built graphs for 24 layers.
Graph size (num tokens): 207
Layer 0: smallest eigenvalue of L = -0.00000000 (should be ~0)
Layer 0: L_sym eigenvalue range = [0.0000, 1.0579] (should be within [0, 2])
Layer 23: smallest eigenvalue of L = -0.00000000 (should be ~0)
Layer 23: L_sym eigenvalue range = [0.0000, 1.1933] (should be within [0, 2])

All Checkpoint 4 assertions passed — graphs are valid at layer 0 and the final layer.


In [19]:
# --- Cell 3: quick visual sanity check — does connectivity look reasonable across layers? ---
fiedler_by_layer = []
for g in layer_graphs:
    eigvals = np.linalg.eigvalsh(g["L"])
    fiedler_by_layer.append(eigvals[1])  # second-smallest = Fiedler value

print("\nFiedler value by layer (quick preview, layer 0 to final):")
for l, fv in enumerate(fiedler_by_layer):
    print(f"  Layer {l:2d}: {fv:.4f}")


Fiedler value by layer (quick preview, layer 0 to final):
  Layer  0: 0.2573
  Layer  1: 0.2109
  Layer  2: 0.1459
  Layer  3: 0.3387
  Layer  4: 0.3146
  Layer  5: 0.3343
  Layer  6: 0.3256
  Layer  7: 0.3162
  Layer  8: 0.1859
  Layer  9: 0.3543
  Layer 10: 0.2942
  Layer 11: 0.3682
  Layer 12: 0.2806
  Layer 13: 0.2892
  Layer 14: 0.2256
  Layer 15: 0.2461
  Layer 16: 0.2749
  Layer 17: 0.3464
  Layer 18: 0.3334
  Layer 19: 0.2568
  Layer 20: 0.3773
  Layer 21: 0.3386
  Layer 22: 0.2862
  Layer 23: 0.2618


In [20]:
# ============================================================
# CHECKPOINT 5 — Spectral diagnostics
# Compute all four primary metrics per layer: Fiedler value,
# spectral entropy, HFER, smoothness (Dirichlet energy kept
# only as a covariate, per the length-confound finding in the pilot).
# ============================================================

In [21]:
# --- Cell 1 ---
import numpy as np

def compute_spectral_metrics(L, hidden_state, eps=1e-10):
    """
    L: Laplacian matrix [seq, seq] (combinatorial, from Checkpoint 4)
    hidden_state: hidden states at this layer [seq, d_model] — the graph
                   SIGNAL used for Dirichlet energy / smoothness / HFER,
                   as distinct from the graph STRUCTURE (L itself).

    Returns dict with: fiedler, spectral_entropy, hfer, smoothness, dirichlet_energy
    """
    n = L.shape[0]

    # Eigendecomposition of L (symmetric -> use eigh, ascending order guaranteed)
    eigvals, eigvecs = np.linalg.eigh(L)
    eigvals = np.clip(eigvals, 0, None)  # guard against tiny negative numerical noise

    # --- Fiedler value: second-smallest eigenvalue ---
    fiedler = eigvals[1] if n > 1 else 0.0

    # --- Spectral entropy: normalize eigenvalues into a distribution, compute Shannon entropy ---
    eigval_sum = eigvals.sum()
    if eigval_sum > eps:
        p = eigvals / eigval_sum
        p_nonzero = p[p > eps]
        spectral_entropy = -np.sum(p_nonzero * np.log(p_nonzero))
        # normalize by log(n) so it's comparable across different sequence lengths
        spectral_entropy = spectral_entropy / np.log(n) if n > 1 else 0.0
    else:
        spectral_entropy = 0.0

    # --- Graph signal for Dirichlet energy / HFER / smoothness: use hidden state norm per token ---
    # (a scalar signal over the graph nodes — standard choice for this framework)
    # (a scalar signal over the graph nodes — standard choice for this framework)
    x = hidden_state.astype(np.float64)  # [seq, d_model]
    signal = np.linalg.norm(x, axis=1)  # [seq]
    signal = (signal - signal.mean()) / (signal.std() + eps)  # standardize

    # --- Dirichlet energy: x^T L x (total variation of the signal w.r.t. graph structure) ---
    dirichlet_energy = float(signal @ L @ signal)

    # --- HFER: project signal onto eigenbasis, compute fraction of energy above median eigenvalue index ---
    signal_proj = eigvecs.T @ signal  # graph Fourier transform
    energy = signal_proj ** 2
    total_energy = energy.sum()
    cutoff_idx = n // 2  # median eigenvalue index, per pilot's method
    hfer = float(energy[cutoff_idx:].sum() / (total_energy + eps))

    # --- Smoothness: normalized inverse of Dirichlet energy ---
    smoothness = float(1.0 / (1.0 + dirichlet_energy))

    return {
        "fiedler": float(fiedler),
        "spectral_entropy": float(spectral_entropy),
        "hfer": hfer,
        "smoothness": smoothness,
        "dirichlet_energy": dirichlet_energy,  # kept as covariate only, per Step 5 decision
    }

In [22]:
# --- Cell 2: run across all layers for the test example, validate ---
all_layer_metrics = []
for l, g in enumerate(layer_graphs):
    hs = result["hidden_states"][l][0].detach().cpu().numpy()  # [seq, d_model]
    metrics = compute_spectral_metrics(g["L"], hs)
    metrics["layer"] = l
    all_layer_metrics.append(metrics)

print(f"{'Layer':>5} {'Fiedler':>10} {'SpecEntropy':>12} {'HFER':>8} {'Smoothness':>11} {'DirichletE':>11}")
for m in all_layer_metrics:
    print(f"{m['layer']:>5} {m['fiedler']:>10.4f} {m['spectral_entropy']:>12.4f} "
          f"{m['hfer']:>8.4f} {m['smoothness']:>11.4f} {m['dirichlet_energy']:>11.4f}")

Layer    Fiedler  SpecEntropy     HFER  Smoothness  DirichletE
    0     0.2573       0.9887   0.5733      0.0049    201.8562
    1     0.2109       0.9824   0.3385      0.0063    157.4062
    2     0.1459       0.9766   0.2255      0.0078    127.5470
    3     0.3387       0.8318   0.9995      0.0001  10697.0710
    4     0.3146       0.8588   0.9996      0.0001   8740.2806
    5     0.3343       0.8341   0.9995      0.0001  10097.5850
    6     0.3256       0.8742   0.9994      0.0001   8438.9580
    7     0.3162       0.7921   0.9997      0.0001  12573.6819
    8     0.1859       0.9095   0.9994      0.0002   6126.2288
    9     0.3543       0.7789   0.9997      0.0001  13568.8053
   10     0.2942       0.8611   0.9994      0.0001   8842.9032
   11     0.3682       0.7946   0.9996      0.0001  13243.2779
   12     0.2806       0.8845   0.9994      0.0001   7606.2929
   13     0.2892       0.8289   0.9996      0.0001  10603.1344
   14     0.2256       0.8910   0.9992      0.0001   67

In [23]:
# --- Cell 3: sanity assertions ---
for m in all_layer_metrics:
    assert 0 <= m["spectral_entropy"] <= 1 + 1e-6, f"STOP: spectral entropy out of [0,1] range at layer {m['layer']}."
    assert 0 <= m["hfer"] <= 1 + 1e-6, f"STOP: HFER out of [0,1] range at layer {m['layer']}."
    assert 0 <= m["smoothness"] <= 1 + 1e-6, f"STOP: smoothness out of [0,1] range at layer {m['layer']}."
    assert m["fiedler"] >= -1e-6, f"STOP: negative Fiedler value at layer {m['layer']}."

print("\nAll Checkpoint 5 assertions passed — metrics are in valid ranges across all 24 layers.")


All Checkpoint 5 assertions passed — metrics are in valid ranges across all 24 layers.


In [24]:
# --- Cell 4: package into a single per-example feature vector (preview of what Checkpoint 8 will do at scale) ---
import pandas as pd
df_preview = pd.DataFrame(all_layer_metrics)
print("\nPreview of the per-layer feature table for this one example:")
print(df_preview.head())


Preview of the per-layer feature table for this one example:
    fiedler  spectral_entropy      hfer  smoothness  dirichlet_energy  layer
0  0.257290          0.988686  0.573272    0.004930        201.856224      0
1  0.210913          0.982425  0.338502    0.006313        157.406170      1
2  0.145920          0.976599  0.225453    0.007779        127.547035      2
3  0.338710          0.831808  0.999450    0.000093      10697.071017      3
4  0.314636          0.858815  0.999636    0.000114       8740.280574      4


In [25]:
# ============================================================
# CHECKPOINT 6 — Dataset generation at scale (GSM8K, 150 examples)
# Resumable: safe to stop and restart without losing completed work.
# ============================================================

In [26]:
# --- Cell 1: load GSM8K ---
from datasets import load_dataset

gsm8k = load_dataset("gsm8k", "main", split="test")
print(f"GSM8K test set size: {len(gsm8k)}")
print("Example:", gsm8k[0])

N_EXAMPLES = 150
import random
random.seed(42)  # fixed seed -> reproducible sample
indices = random.sample(range(len(gsm8k)), N_EXAMPLES)
subset = gsm8k.select(indices)
print(f"\nSelected {N_EXAMPLES} examples (seed=42, reproducible).")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K test set size: 1319
Example: {'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}

Selected 150 examples (seed=42, reproducible).


In [27]:
# --- Cell 2: resumable save/load helpers ---
import os, json, re, hashlib

GSM8K_DIR = f"{DIRS['generations']}/gsm8k"
os.makedirs(GSM8K_DIR, exist_ok=True)

def example_id(question_text):
    """Content-based hash -> stable ID even if ordering changes.
    (Your pilot's fix for the stale-example_id collision bug.)"""
    return hashlib.sha256(question_text.encode()).hexdigest()[:12]

def already_done(ex_id):
    return os.path.exists(f"{GSM8K_DIR}/{ex_id}.json")

def save_example(ex_id, data):
    # Write to a temp file then rename -> atomic, avoids corrupt partial writes
    # if the session dies mid-write.
    tmp_path = f"{GSM8K_DIR}/{ex_id}.json.tmp"
    final_path = f"{GSM8K_DIR}/{ex_id}.json"
    with open(tmp_path, "w") as f:
        json.dump(data, f)
    os.rename(tmp_path, final_path)

def extract_gsm8k_answer(text):
    """GSM8K ground-truth answers are formatted '#### <number>'."""
    match = re.search(r"####\s*([\-0-9,.]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()
    return None

def extract_model_answer(generated_text):
    """Pull the last number-like token from the model's response as its final answer.
    Simple heuristic — good enough for auto-grading, spot-check a sample after."""
    numbers = re.findall(r"[\-0-9,]*\.?[0-9]+", generated_text.replace(",", ""))
    return numbers[-1] if numbers else None

In [28]:
# --- Cell 3: the resumable generation loop ---
SAVE_EVERY = 1  # save after every example (cheap, safest for a 4-day deadline)

completed = 0
skipped = 0
failed = 0

for i, ex in enumerate(subset):
    question = ex["question"]
    ground_truth_raw = ex["answer"]
    ground_truth = extract_gsm8k_answer(ground_truth_raw)
    ex_id = example_id(question)

    if already_done(ex_id):
        skipped += 1
        continue

    try:
        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": f"{question}\n\nThink step by step, then give your final numeric answer."}],
            tokenize=False, add_generation_prompt=True,
        )

        gen_result = generate_and_replay(prompt, model, tokenizer, max_new_tokens=512, temperature=0.3)

        response_text = tokenizer.decode(
            gen_result["full_input_ids"][0][gen_result["prompt_len"]:], skip_special_tokens=True
        )
        model_answer = extract_model_answer(response_text)
        is_correct = (model_answer == ground_truth) if model_answer is not None else False

        # Save attention/hidden-state tensors separately (they're large) -
        # only save what's needed: per-layer W matrices from Checkpoint 4,
        # or raw attentions if you want to rebuild graphs later. Saving
        # raw hidden states + attentions for 150 examples x 24 layers can
        # get large, so we save them as compressed numpy here.
        import numpy as np
        tensor_path = f"{GSM8K_DIR}/{ex_id}_tensors.npz"
        attn_stack = np.stack([a[0].cpu().numpy() for a in gen_result["attentions"]])  # [layers, heads, seq, seq]
        hs_stack = np.stack([h[0].cpu().numpy() for h in gen_result["hidden_states"]])  # [layers+1, seq, d_model]
        np.savez_compressed(tensor_path, attentions=attn_stack.astype(np.float16), hidden_states=hs_stack.astype(np.float16))

        record = {
            "example_id": ex_id,
            "dataset": "gsm8k",
            "question": question,
            "ground_truth": ground_truth,
            "response_text": response_text,
            "model_answer": model_answer,
            "is_correct": is_correct,
            "prompt_len": gen_result["prompt_len"],
            "response_len": gen_result["response_len"],
            "total_len": gen_result["total_len"],
            "tensor_path": tensor_path,
        }
        save_example(ex_id, record)
        completed += 1

        if (i + 1) % 10 == 0:
            print(f"[{i+1}/{N_EXAMPLES}] completed={completed} skipped={skipped} failed={failed}")

    except Exception as e:
        failed += 1
        print(f"FAILED on example {i} ({ex_id}): {e}")
        # Don't crash the whole loop on one bad example -- log and continue.
        continue

print(f"\nDone. completed={completed} skipped(already done)={skipped} failed={failed}")
print(f"Total examples on disk: {len(os.listdir(GSM8K_DIR)) // 2}")  # /2 since each has .json + tensors.npz

[10/150] completed=10 skipped=0 failed=0
[20/150] completed=20 skipped=0 failed=0
[30/150] completed=30 skipped=0 failed=0
[40/150] completed=40 skipped=0 failed=0
[50/150] completed=50 skipped=0 failed=0
[60/150] completed=60 skipped=0 failed=0
[70/150] completed=70 skipped=0 failed=0
[80/150] completed=80 skipped=0 failed=0
[90/150] completed=90 skipped=0 failed=0
[100/150] completed=100 skipped=0 failed=0
[110/150] completed=110 skipped=0 failed=0
[120/150] completed=120 skipped=0 failed=0
[130/150] completed=130 skipped=0 failed=0
[140/150] completed=140 skipped=0 failed=0
[150/150] completed=150 skipped=0 failed=0

Done. completed=150 skipped(already done)=0 failed=0
Total examples on disk: 150
